Pre-processing of online_retail_II dataset

In [1]:
import pandas as pd

In [2]:
file_path = "data/online_retail_II.xlsx"
retail_excel = pd.ExcelFile(file_path)

In [3]:
retail_excel.sheet_names

['Year 2009-2010', 'Year 2010-2011']

In [4]:
retail_2009_2010 = pd.read_excel(
    file_path,
    sheet_name="Year 2009-2010")


retail_2010_2011 = pd.read_excel(
    file_path,
    sheet_name="Year 2010-2011")

In [5]:
retail_2009_2010.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [6]:
# Joining both sheets into one dataset

retail_transactions = pd.concat(
    [retail_2009_2010, retail_2010_2011],
    ignore_index=True)

In [7]:
retail_transactions.info()

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[us]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 65.1+ MB


Roughly 243,000 transactions do not have a customer ID, this is worth looking into

In [8]:
# Missing Values

retail_transactions.isnull().sum()

Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

In [9]:
# Investigate missing values further

retail_transactions[retail_transactions["Description"].isnull()].head(10)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.0,NaN,United Kingdom
3114,489655,20683,NaN,-44,2009-12-01 17:26:00,0.0,NaN,United Kingdom
3161,489659,21350,NaN,230,2009-12-01 17:39:00,0.0,NaN,United Kingdom
3731,489781,84292,NaN,17,2009-12-02 11:45:00,0.0,NaN,United Kingdom
4296,489806,18010,NaN,-770,2009-12-02 12:42:00,0.0,NaN,United Kingdom
4566,489821,85049G,NaN,-240,2009-12-02 13:25:00,0.0,NaN,United Kingdom
6378,489882,35751C,NaN,12,2009-12-02 16:22:00,0.0,NaN,United Kingdom
6555,489898,79323G,NaN,954,2009-12-03 09:40:00,0.0,NaN,United Kingdom
6576,489901,21098,NaN,-200,2009-12-03 09:47:00,0.0,NaN,United Kingdom
6581,489903,21166,NaN,48,2009-12-03 09:57:00,0.0,NaN,United Kingdom


In [10]:
retail_transactions[
    retail_transactions["Customer ID"].isnull()].head(10)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.00,NaN,United Kingdom
283,489463,71477,short,-240,2009-12-01 10:52:00,0.00,NaN,United Kingdom
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.00,NaN,United Kingdom
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.00,NaN,United Kingdom
577,489525,85226C,BLUE PULL BACK RACING CAR,1,2009-12-01 11:49:00,0.55,NaN,United Kingdom
578,489525,85227,SET/6 3D KIT CARDS FOR KIDS,1,2009-12-01 11:49:00,0.85,NaN,United Kingdom
1055,489548,22271,FELTCRAFT DOLL ROSIE,1,2009-12-01 12:32:00,2.95,NaN,United Kingdom
1056,489548,22254,FELT TOADSTOOL LARGE,12,2009-12-01 12:32:00,1.25,NaN,United Kingdom
1057,489548,22273,FELTCRAFT DOLL MOLLY,3,2009-12-01 12:32:00,2.95,NaN,United Kingdom
1058,489548,22195,LARGE HEART MEASURING SPOONS,1,2009-12-01 12:32:00,1.65,NaN,United Kingdom


In [11]:
# We can drop rows with no description as they only acoount for a fraction of the dataset.

retail_transactions = retail_transactions.dropna(subset=["Description"])

The rows with missing customerIDs will be kept in the dataset, a missing Customer ID doesn't
prevent a row from being used and will simply need to be accepted as a limitation of the dataset.

While being a limitation, it should not have a significant impact on the analysis.

In [12]:
# Cancelled transaction have an invoice number beginning with C, making a separate column for them:

retail_transactions["is_cancelled"] = (retail_transactions["Invoice"].astype(str).str.startswith("C"))

In [13]:
retail_transactions["is_cancelled"].value_counts()

is_cancelled
False    1043495
True       19494
Name: count, dtype: int64

In [14]:
retail_transactions[
    retail_transactions["is_cancelled"] == True
].head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,is_cancelled
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321.0,Australia,True
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321.0,Australia,True
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321.0,Australia,True
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321.0,Australia,True
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321.0,Australia,True


In [15]:
# Creating a revenue column for better analysis

retail_transactions["Revenue"] = (retail_transactions["Quantity"]*retail_transactions["Price"])

In [16]:
retail_transactions['Revenue'] = retail_transactions['Revenue'].round(2)

In [17]:
retail_transactions[["Quantity", "Price", "Revenue"]].head()

,Quantity,Price,Revenue
0,12,6.95,83.4
1,12,6.75,81.0
2,12,6.75,81.0
3,48,2.10,100.8
4,24,1.25,30.0


In [18]:
retail_transactions["Customer ID"].isna().sum()

np.int64(238625)

Improving dataset timing, for this analysis the exact time is not needed, only the date:

In [19]:
retail_transactions["YearMonth"] = retail_transactions["InvoiceDate"].dt.to_period("M")

In [ ]:
# Creating a separate dataframe for normal sales without cancellations

sales_transactions = retail_transactions[retail_transactions["is_cancelled"] == False]

In [21]:
sales_transactions = sales_transactions.drop(columns=["is_cancelled"])

In [22]:
sales_transactions.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue,YearMonth
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.4,2009-12
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0,2009-12
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0,2009-12
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,100.8,2009-12
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0,2009-12


Checking for negative quantity rows

In [23]:
neg_qty = sales_transactions[sales_transactions["Quantity"] < 0]
print(f"Negative quantity, non-cancelled rows: {len(neg_qty)}")
neg_qty[["Invoice", "StockCode", "Description", "Quantity", "Price", "Customer ID"]].head(20)

Negative quantity, non-cancelled rows: 768


,Invoice,StockCode,Description,Quantity,Price,Customer ID
263,489464,21733,85123a mixed,-96,0.0,NaN
283,489463,71477,short,-240,0.0,NaN
284,489467,85123A,21733 mixed,-192,0.0,NaN
3162,489660,35956,lost,-1043,0.0,NaN
3168,489663,35605A,damages,-117,0.0,NaN
4538,489820,21133,invcd as 84879?,-720,0.0,NaN
6556,489899,79323GR,sold as gold,-954,0.0,NaN
6911,490007,84347,21494,-720,0.0,NaN
9308,490130,21493,lost?,-600,0.0,NaN
17427,490765,21450,damaged,-31,0.0,NaN


Checking for any price values set to 0

In [24]:
zero_price = sales_transactions[sales_transactions["Price"] == 0]
print(f"Price = 0 rows: {len(zero_price)}")
zero_price[["Invoice", "StockCode", "Description", "Quantity", "Customer ID"]].head(20)

Price = 0 rows: 1820


,Invoice,StockCode,Description,Quantity,Customer ID
263,489464,21733,85123a mixed,-96,NaN
283,489463,71477,short,-240,NaN
284,489467,85123A,21733 mixed,-192,NaN
3162,489660,35956,lost,-1043,NaN
3168,489663,35605A,damages,-117,NaN
4538,489820,21133,invcd as 84879?,-720,NaN
4674,489825,22076,6 RIBBONS EMPIRE,12,16126.0
5904,489861,DOT,DOTCOM POSTAGE,1,NaN
6556,489899,79323GR,sold as gold,-954,NaN
6781,489998,48185,DOOR MAT FAIRY CAKE,2,15658.0


There seems to be a pattern of customerIDs being missing for these orders, investigating further:

In [36]:
neg_qty = sales_transactions[sales_transactions["Quantity"] < 0]
print(neg_qty["Customer ID"].isna().value_counts())
print(neg_qty["Description"].value_counts().head(20))

zero_price = sales_transactions[sales_transactions["Price"] == 0]
print(zero_price["Customer ID"].isna().value_counts())
print(zero_price["Description"].value_counts().head(20))

Customer ID
True    625
Name: count, dtype: int64
Description
damages                   84
?                         83
damaged                   78
missing                   27
sold as set on dotcom     20
Damaged                   17
smashed                    9
thrown away                9
Unsaleable, destroyed.     9
damages?                   7
??                         7
crushed                    6
given away                 6
MIA                        5
Damages                    5
counted                    5
checked                    5
wet damaged                5
ebay                       5
ebay sales                 4
Name: count, dtype: int64
Customer ID
True     1552
False      82
Name: count, dtype: int64
Description
?                                      92
damages                                84
damaged                                81
found                                  28
missing                                27
sold as set on dotcom                  20
PA

Negative-quantity rows (625) all lack a Customer ID and their descriptions ("damages", "missing",
"thrown away", "unsaleable, destroyed") show they are internal stock adjustments, not customer
transactions bypassing the cancellation flag. These will be dropped.

Price = 0 rows have a similar pattern: 1,552 of 1,634 also lack a Customer ID and show the same
adjustment-style descriptions, and are dropped for the same reason. The remaining 82 rows have
a genuine Customer ID and are kept, these likely represent free items or promotional add ons
attached to real orders, where zero price doesn't indicate a data quality issue.

In [37]:
sales_transactions = sales_transactions[
    (sales_transactions["Quantity"] > 0) &
    ~((sales_transactions["Price"] == 0) & (sales_transactions["Customer ID"].isna()))
]
print(f"Rows remaining: {len(sales_transactions)}")

Rows remaining: 1037526


In [25]:
sales_transactions["Price"] = pd.to_numeric(
    sales_transactions["Price"], 
    errors="coerce"
)

sales_transactions["Price"] = sales_transactions["Price"].round(2)

Checking for exact duplicate line items (same invoice, stock code, quantity, date, and price)

In [26]:
key_cols = ["Invoice", "StockCode", "Quantity", "InvoiceDate", "Price"]
key_dupes = sales_transactions[sales_transactions.duplicated(subset=key_cols, keep=False)]
print(len(key_dupes))
key_dupes.sort_values(key_cols).head(20)

66258


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue,YearMonth
379,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom,1.95,2009-12
391,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom,1.95,2009-12
365,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,3.75,2009-12
386,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,3.75,2009-12
363,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,3.75,2009-12
371,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,3.75,2009-12
394,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,3.75,2009-12
362,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,3.75,2009-12
385,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,3.75,2009-12
368,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,2009-12-01 11:34:00,0.85,16329.0,United Kingdom,5.10,2009-12


These could be genuine repeat additions of the same item within an order rather than data errors, and
there's no way to distinguish the two from the available columns. Leaving these rows in place
rather than dropping them, to avoid removing real purchases.

# Using keywords to group together products based on given description

Product descriptions were grouped into business-relevant categories using a rule-based keyword classification approach. Keywords were developed by reviewing high-revenue products and common product-description patterns. This provides a consistent categorisation for category-level analysis while retaining an `Other` category for products that could not be confidently assigned.

AI assistance was used to efficiently review and identify patterns across 1,000,000+ rows of unstructured product descriptions, supporting the development of relevant keywords and improving the efficiency of the categorisation workflow. The resulting classification was implemented as a reproducible rule-based approach in Python.

In [27]:
top_descriptions = (
    sales_transactions.groupby('Description')['Revenue']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)
print(top_descriptions.head(5000))

                              Description    Revenue
0                REGENCY CAKESTAND 3 TIER  344563.25
1                                  Manual  340731.33
2                          DOTCOM POSTAGE  322657.48
3      WHITE HANGING HEART T-LIGHT HOLDER  266923.55
4             PAPER CRAFT , LITTLE BIRDIE  168469.60
...                                   ...        ...
4995  SET OF 4 NAPKIN CHARMS INSTUMENT         30.60
4996    Electronic Talking Breath-A-Loser      30.47
4997   BAROQUE BUTTERFLY EARRINGS CRYSTAL      30.23
4998     ASSORTED FRAGRANCE BATH CONFETTI      30.16
4999    BLUE/YELLOW CERAMIC CANDLE HOLDER      30.08

[5000 rows x 2 columns]


In [28]:
# Export top 5000 descriptions as CSV

# top_descriptions.head(5000).to_csv('top_5000_descriptions.csv', index=False)

In [29]:
# Remove administrative/non-product entries identified

non_products = [
    'MANUAL',
    'POSTAGE',
    'DOTCOM POSTAGE',
    'DOTCOM',
    'AMAZON FEE',
    'CHECK',
    'AMAZON ADJUSTMENT',
    'RE-ADJUSTMENT',
    'CHECK?',
    'STOCK CHECK',
    'SALE ERROR',
    'Adjust bad debt'
]

sales_transactions = sales_transactions[
    ~sales_transactions['Description'].str.upper().isin(
        [x.upper() for x in non_products])]



# Category keyword dictionary - AI-assisted, built from top 200 products by revenue
CATEGORY_KEYWORDS = {

    'Christmas & Seasonal': [
        'CHRISTMAS', 'XMAS', 'SANTA', 'REINDEER', 'ADVENT', 'SLEIGH',
        'SNOWMAN', 'SNOWFLAKE', 'EASTER', 'HALLOWEEN', 'PUMPKIN', 'DECORATION'
    ],

    'Lighting & Candles': [
        'LIGHT', 'CANDLE', 'LANTERN', 'STRING', 'LED', 'GLOW', 'FLAME'
    ],

    'Bags & Totes': [
        'BAG', 'TOTE', 'SHOPPER', 'POUCH', 'LUNCH'
    ],

    'Kitchen & Baking': [
        'CAKE', 'KITCHEN', 'TEA', 'BREAD', 'BISCUIT', 'SPICE',
        'BAKING', 'JAM', 'JELLY', 'ENAMEL', 'COOKIE', 'EGG',
        'FRYING', 'CUTLERY', 'PANTRY', 'RECIPE', 'MUG', 'SAUCER',
        'POPCORN', 'SCALES', 'TIN', 'JAR', 'BOWL', 'SNACK', 'GLASS',
        'CERAMIC', 'PORCELAIN', 'TRAY', 'RACK', 'FLASK', 'COFFEE'
    ],

    'Home Decor & Storage': [
        'FRAME', 'BOARD', 'CABINET', 'DRAWER', 'SHELF', 'COAT',
        'STORAGE', 'SIGN', 'DOOR', 'CLOCK', 'CRATE', 'HOOK',
        'CHEST', 'BLOCK', 'MIRROR', 'WALL', 'PLAQUE', 'STAND', 'HANGER',
        'CASE', 'CASES'
    ],

    'Gift Wrap & Party': [
        'BUNTING', 'PAPER', 'RIBBON', 'GIFT', 'TISSUE',
        'WRAP', 'NAPKIN', 'CARD'
    ],

    'Home Comfort & Textiles': [
        'HOT WATER', 'HOTTIE', 'HAND WARMER',
        'CUSHION', 'THROW', 'BLANKET', 'KNITTED'
    ],

    'Novelty & Gifts': [
        'TRINKET', 'HARMONICA', 'PLASTER', 'SILK', 'BABUSHKA',
        'ALPHABET', 'WICKER', 'ORNAMENT', 'ZINC', 'MINIATURE', 'BELL',
        'NECKLACE', 'BRACELET', 'EARRING', 'BANGLE', 'BROOCH',
        'PENDANT', 'HAIR', 'DIAMANTE', 'CRYSTAL', 'RING', 'LOCKET',
        'UNION', 'LONDON', 'SCOTTIE', 'SPACEBOY', 'RETROSPOT',
        'SUKI', 'WOODLAND', 'PAISLEY', 'SKULL', 'SCANDINAVIAN',
        'BAROQUE', 'DOILY', 'POLKADOT', 'SPOTTY', 'DOLLY GIRL', 'STRAWBERRY'
    ],

    'Craft & Art': [
        'FELTCRAFT', 'CRAFT', 'PAINT', 'IRON ON', 'SEWING',
        'MODELLING', 'COLOURING', 'STENCIL', 'KNITTING', 'STAMP',
        'NEEDLE', 'NOTEBOOK', 'PEN', 'PENCIL', 'HIGHLIGHTER', 'STATIONERY'
    ],

    'Garden & Nature': [
        'GARDEN', 'PLANT', 'GROW', 'SEED', 'TROWEL', 'KNEELING',
        'CROQUET', 'ROUNDERS', 'SKITTLE', 'PICNIC', 'UMBRELLA', 'PARASOL',
        'BIRD', 'BUTTERFLY', 'TOADSTOOL', 'DOVE', 'MUSHROOM',
        'FLORAL', 'FLOWER', 'ROSE', 'BOTANICAL', 'LEAF', 'OWL',
        'FOX', 'HEDGEHOG', 'ANIMAL',
        'SKIPPING', 'JIGSAW', 'DOMINO', 'PUZZLE', 'INFLATABLE',
        'GLIDER', 'DOLL', 'SNAKES', 'GAME', 'TOY', 'PUPPET', 'BALL'
    ],

}

def assign_category(description):
    if pd.isna(description) or not isinstance(description, str):
        return 'Unknown'
    desc = description.upper()
    for category, keywords in CATEGORY_KEYWORDS.items():
        if any(keyword in desc for keyword in keywords):
            return category
    return 'Other'

sales_transactions['Category'] = sales_transactions['Description'].apply(assign_category)

In [30]:
# Checking unknown category

print(sales_transactions[sales_transactions['Category'] == 'Unknown'])

       Invoice StockCode Description  Quantity         InvoiceDate  Price  \
6911    490007     84347       21494      -720 2009-12-03 12:09:00    0.0   
274052  516016     22467       22719         2 2010-07-16 10:11:00    0.0   
274053  516017     22719       22467        -2 2010-07-16 10:11:00    0.0   
945851  572891     23343       20713      -400 2011-10-26 14:14:00    0.0   

        Customer ID         Country  Revenue YearMonth Category  
6911            NaN  United Kingdom     -0.0   2009-12  Unknown  
274052          NaN  United Kingdom      0.0   2010-07  Unknown  
274053          NaN  United Kingdom     -0.0   2010-07  Unknown  
945851          NaN  United Kingdom     -0.0   2011-10  Unknown  


In [31]:
# dropping the uknown category rows above, as they are not useful for analysis and there are only 4

sales_transactions = sales_transactions[sales_transactions['Category'] != 'Unknown']

In [32]:
sales_transactions.groupby('Category')['Quantity'].sum().sort_values(ascending=False)

Category
Kitchen & Baking           2526324
Home Decor & Storage       1285855
Gift Wrap & Party          1258587
Bags & Totes               1235469
Novelty & Gifts            1126802
Lighting & Candles         1082628
Christmas & Seasonal        888686
Other                       715563
Garden & Nature             492619
Craft & Art                 395063
Home Comfort & Textiles     165567
Name: Quantity, dtype: int64

In [33]:
category_revenue = (
    sales_transactions.groupby('Category')['Revenue']
    .sum()
    .sort_values(ascending=False)
)
total_revenue = category_revenue.sum()
for cat, rev in category_revenue.items():
    print(f"{cat}: £{rev:,.0f} ({rev/total_revenue*100:.1f}%)")

other_pct = category_revenue.get('Other', 0) / total_revenue * 100
print(f"\nOther accounts for {other_pct:.1f}% of total revenue")

Kitchen & Baking: £5,286,062 (26.3%)
Home Decor & Storage: £3,384,446 (16.8%)
Bags & Totes: £2,186,119 (10.9%)
Lighting & Candles: £1,899,795 (9.4%)
Novelty & Gifts: £1,601,571 (8.0%)
Other: £1,285,520 (6.4%)
Christmas & Seasonal: £1,235,359 (6.1%)
Gift Wrap & Party: £1,149,734 (5.7%)
Garden & Nature: £886,786 (4.4%)
Home Comfort & Textiles: £714,931 (3.6%)
Craft & Art: £506,214 (2.5%)

Other accounts for 6.4% of total revenue


Final Checks

In [34]:
print(f"Final row count: {len(sales_transactions)}")
print()
print("Nulls remaining:")
print(sales_transactions.isnull().sum())
print()
print("Category distribution (% of revenue):")
print((sales_transactions.groupby('Category')['Revenue'].sum() / sales_transactions['Revenue'].sum() * 100).sort_values(ascending=False).round(1))

Final row count: 1039078

Nulls remaining:
Invoice             0
StockCode           0
Description         0
Quantity            0
InvoiceDate         0
Price               0
Customer ID    236028
Country             0
Revenue             0
YearMonth           0
Category            0
dtype: int64

Category distribution (% of revenue):
Category
Kitchen & Baking           26.3
Home Decor & Storage       16.8
Bags & Totes               10.9
Lighting & Candles          9.4
Novelty & Gifts             8.0
Other                       6.4
Christmas & Seasonal        6.1
Gift Wrap & Party           5.7
Garden & Nature             4.4
Home Comfort & Textiles     3.6
Craft & Art                 2.5
Name: Revenue, dtype: float64


In [ ]:
import csv

# sales_transactions.to_csv("cleaned_sales_transactions.csv", index=False, quoting=csv.QUOTE_MINIMAL)